# PCB Defect Detection — Colab Eğitim Defteri (CMPE 7144)
Akış: **GitHub'dan kodu çek → Drive'dan veriyi kopyala → eğit → çıktıları Drive'a kaydet.**

Başlamadan önce:
1. *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*
2. Sol menüdeki **🔑 Gizli anahtarlar (Secrets)** bölümüne `GITHUB_TOKEN` adıyla GitHub erişim jetonunu ekle (depo özelse).
3. `DsPCBSD+.zip` dosyasını Drive'da `MyDrive/CMPE7144_PCB/data/` klasörüne yükle (~123 MB, bir kez).

In [ ]:
!nvidia-smi

In [ ]:
# --- Ayarlar ---
GITHUB_USER = "KULLANICI_ADIN"          # değiştir
REPO_NAME   = "pcb-surface-defect-detection"
DRIVE_ROOT  = "/content/drive/MyDrive/CMPE7144_PCB"
DATA_ZIP    = f"{DRIVE_ROOT}/data/DsPCBSD+.zip"
RUNS_DIR    = f"{DRIVE_ROOT}/runs"        # checkpoint'ler Drive'a yazılır, oturum kopsa da kaybolmaz
MODEL       = "yolo11s.pt"
RUN_NAME    = "yolo11s_640_seed0"
EPOCHS, IMGSZ, BATCH, SEED = 100, 640, 32, 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# --- Kodu GitHub'dan çek ---
import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"   # depo herkese açıksa
if os.path.exists(f"/content/{REPO_NAME}"):
    !git -C /content/{REPO_NAME} pull
else:
    !git clone {url} /content/{REPO_NAME}
%cd /content/{REPO_NAME}
!pip -q install -r requirements.txt

In [ ]:
# --- Veriyi Drive'dan yerel diske kopyala ve aç (Drive'dan doğrudan okumak çok yavaştır) ---
import glob, os
if not glob.glob("/content/DsPCBSD/**/Data_YOLO", recursive=True):
    !cp "{DATA_ZIP}" /content/dspcbsd.zip
    !unzip -q /content/dspcbsd.zip -d /content/DsPCBSD
yolo_root = glob.glob("/content/DsPCBSD/**/Data_YOLO", recursive=True)[0]
print("YOLO verisi:", yolo_root)
!ls {yolo_root}/images

In [ ]:
# --- Veri seti yaml'ını Colab yollarıyla yaz ---
import yaml
cfg = yaml.safe_load(open("configs/dspcbsd.yaml"))
cfg["path"] = yolo_root
yaml.safe_dump(cfg, open("/content/dspcbsd_colab.yaml", "w"), sort_keys=False)
print(open("/content/dspcbsd_colab.yaml").read())

In [ ]:
# --- Eğitim (yarım kalan eğitim varsa kaldığı yerden devam eder) ---
from ultralytics import YOLO
last = f"{RUNS_DIR}/{RUN_NAME}/weights/last.pt"
if os.path.exists(last):
    model = YOLO(last); model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(data="/content/dspcbsd_colab.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                seed=SEED, project=RUNS_DIR, name=RUN_NAME, exist_ok=True, plots=True)

In [ ]:
# --- Doğrulama setinde değerlendirme (sınıf bazlı sonuçlar + COCO json) ---
best = YOLO(f"{RUNS_DIR}/{RUN_NAME}/weights/best.pt")
metrics = best.val(data="/content/dspcbsd_colab.yaml", split="val", imgsz=IMGSZ,
                   save_json=True, project=RUNS_DIR, name=f"{RUN_NAME}_val", exist_ok=True)
print("mAP@0.5:", metrics.box.map50, " mAP@0.5:0.95:", metrics.box.map)